# 🏗️ CEM4644 · MP4 — Segmentation for progress and quantity measurement
## Homework (individual): *Interior finishing: drywall, studs, insulation, pipes, tiles*

**No coding needed.** Each grey box below is one *step*: click the ▶ (play) button at its left, wait until it finishes, look at the result, then answer the report question that follows. Run the steps **from top to bottom**.

**What you will do (about 120 minutes)**
1. Look at site photos and the materials in them.
2. Ask a segmentation model, by name, for a material: see the mask, the overlay, and the share of the photo it covers.
3. Compare photos and follow a site through time.
4. Examine where the model goes wrong: wording, weak regions, and how to correct it.
5. Measure footing areas on a structural plan (quantity take-off).

**Before you start:** menu *Runtime → Change runtime type → T4 GPU → Save*. The model used here (SAM 3) is large: with a GPU each request takes well under a second; without one, precomputed results still work but live requests take about a minute each.

In [ ]:
#@title ▶ Step 0 · Run me first (2–3 minutes) { display-mode: "form" }
#@markdown Click ▶ and wait for the green ✅ line. This downloads the photos with their precomputed results and loads SAM 3 (about 3 GB).
#@markdown Untick *load_model* only if you have no GPU and want to skip the live steps.
load_model = True #@param {type:"boolean"}
import os, sys, subprocess
if not os.path.isdir("CEM4644/mp4_segmentation"):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/Haolan-Zhang/CEM4644.git"], check=True)
sys.path.insert(0, os.path.abspath("CEM4644/mp4_segmentation"))
from aec_seg import lab
lab.setup(dataset="interior", load_model=load_model, plans=["plan_whittier", "plan_mill"])


## Part 1 · Meet the photos

Detection (MP3) draws a **box** around an object. **Segmentation** goes one step further: it decides, *pixel by pixel*, what belongs to the object. That is what makes it useful for measurement: count the pixels of a material and you have its share of the photo; know the scale and you have an area.

The model in this notebook is **SAM 3** (Segment Anything Model 3, Meta 2025). You do not train it. You type a short phrase, such as *concrete* or *steel reinforcement bars*, and it returns every region in the photo that matches, each with a **confidence**.

Photos of interior fit-out: drywall (plasterboard) being installed, stud walls, insulation, services and tiling, plus three 1938 interiors from the Belchertown project record. All from Wikimedia Commons under open licences.

In [ ]:
#@title ▶ Step 1a · Browse the photos { display-mode: "form" }
#@markdown *all* shows every photo with its credit. A series letter shows one site through time.
which = "all" #@param ["all", "C"]
lab.show_photos(which)


## Part 2 · Segment a material by name

Pick a photo and a material. You get three panels: the photo, the **mask** (white = the model says *this is it*), and the **overlay**. The share of the photo covered by the mask is printed below, together with the confidence of each region. The **confidence slider** hides the regions the model is unsure about: watch how the share changes.

In [ ]:
#@title ▶ Step 2a · Original → mask → overlay { display-mode: "form" }
#@markdown Try several materials on the same photo, then the same material on other photos.
photo = "int_19: Seabees renovate office spaces, Osan (1)" #@param ["int_01: Drywall crane and workers", "int_06: Gypsum board interior wall, Elm Place", "int_07: House ready for drywall (studs and insulation)", "int_09: Drywall penetrations in an industrial building", "int_10: Drywall firestop detail", "int_11: Belchertown: boiler room, 14 Jan 1938", "int_12: Belchertown: plumbing in the basement toilet, 10 Mar 1938", "int_13: Belchertown: tiling in the first-floor toilet, 10 Jun 1938", "int_15: Drywall shaft damage", "int_16: Hanging drywall", "int_17: Carpenters installing sheetrock, LIRR Concourse, 2019", "int_18: Drywall installed in an office hallway, Canaan Valley", "int_19: Seabees renovate office spaces, Osan (1)", "int_20: Seabees renovate office spaces, Osan (2)", "int_21: A builder marks sheetrock", "int_22: Builders cut sheetrock", "int_23: Piping insulation on ventilation pipes, LIRR Concourse, 2019", "int_24: Seabees small-scale construction project, Jamaica, 2024 (1)", "int_25: Seabees small-scale construction project, Jamaica, 2024 (2)"]
material = "drywall (plasterboard)" #@param ["drywall (plasterboard)", "studs / framing", "insulation", "pipes / ducts", "tiles", "concrete", "ceiling", "floor", "windows / openings", "worker"]
threshold = 0.5 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
lab.segment(photo, material, threshold)


In [ ]:
#@title ▶ Step 2b · The material mix of one photo { display-mode: "form" }
#@markdown Every material at once, with the share of the photo each one covers. Regions may overlap (a worker standing in front of concrete), so the shares need not add up to 100.
photo = "int_19: Seabees renovate office spaces, Osan (1)" #@param ["int_01: Drywall crane and workers", "int_06: Gypsum board interior wall, Elm Place", "int_07: House ready for drywall (studs and insulation)", "int_09: Drywall penetrations in an industrial building", "int_10: Drywall firestop detail", "int_11: Belchertown: boiler room, 14 Jan 1938", "int_12: Belchertown: plumbing in the basement toilet, 10 Mar 1938", "int_13: Belchertown: tiling in the first-floor toilet, 10 Jun 1938", "int_15: Drywall shaft damage", "int_16: Hanging drywall", "int_17: Carpenters installing sheetrock, LIRR Concourse, 2019", "int_18: Drywall installed in an office hallway, Canaan Valley", "int_19: Seabees renovate office spaces, Osan (1)", "int_20: Seabees renovate office spaces, Osan (2)", "int_21: A builder marks sheetrock", "int_22: Builders cut sheetrock", "int_23: Piping insulation on ventilation pipes, LIRR Concourse, 2019", "int_24: Seabees small-scale construction project, Jamaica, 2024 (1)", "int_25: Seabees small-scale construction project, Jamaica, 2024 (2)"]
threshold = 0.5 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
lab.material_mix(photo, threshold)


In [ ]:
#@title ▶ Step 2c · Can *you* estimate the share? { display-mode: "form" }
#@markdown A photo and a material: guess how much of the photo it covers, then see what SAM 3 measures.
rounds = 4 #@param {type:"slider", min:2, max:8, step:1}
lab.guess_game(rounds)


> ### 📝 Report question 1
> What was your score in the estimation game? Pick one photo and give its material mix at confidence 0.5 (the numbers from Step 2b). Then move the threshold to 0.3 and 0.8 for one material: how much does the share change, and why?

## Part 3 · Compare photos and follow progress

Two photos of the same site at different times tell a story: formwork and rebar disappear, concrete and brick appear. Comparing the material shares turns that story into numbers. Keep in mind what the number is: the share of the **photo**, not the share of the **work**: camera position, zoom and the sky change it without any progress on site.

In [ ]:
#@title ▶ Step 3a · Compare two photos { display-mode: "form" }
photo_a = "int_16: Hanging drywall" #@param ["int_01: Drywall crane and workers", "int_06: Gypsum board interior wall, Elm Place", "int_07: House ready for drywall (studs and insulation)", "int_09: Drywall penetrations in an industrial building", "int_10: Drywall firestop detail", "int_11: Belchertown: boiler room, 14 Jan 1938", "int_12: Belchertown: plumbing in the basement toilet, 10 Mar 1938", "int_13: Belchertown: tiling in the first-floor toilet, 10 Jun 1938", "int_15: Drywall shaft damage", "int_16: Hanging drywall", "int_17: Carpenters installing sheetrock, LIRR Concourse, 2019", "int_18: Drywall installed in an office hallway, Canaan Valley", "int_19: Seabees renovate office spaces, Osan (1)", "int_20: Seabees renovate office spaces, Osan (2)", "int_21: A builder marks sheetrock", "int_22: Builders cut sheetrock", "int_23: Piping insulation on ventilation pipes, LIRR Concourse, 2019", "int_24: Seabees small-scale construction project, Jamaica, 2024 (1)", "int_25: Seabees small-scale construction project, Jamaica, 2024 (2)"]
photo_b = "int_18: Drywall installed in an office hallway, Canaan Valley" #@param ["int_01: Drywall crane and workers", "int_06: Gypsum board interior wall, Elm Place", "int_07: House ready for drywall (studs and insulation)", "int_09: Drywall penetrations in an industrial building", "int_10: Drywall firestop detail", "int_11: Belchertown: boiler room, 14 Jan 1938", "int_12: Belchertown: plumbing in the basement toilet, 10 Mar 1938", "int_13: Belchertown: tiling in the first-floor toilet, 10 Jun 1938", "int_15: Drywall shaft damage", "int_16: Hanging drywall", "int_17: Carpenters installing sheetrock, LIRR Concourse, 2019", "int_18: Drywall installed in an office hallway, Canaan Valley", "int_19: Seabees renovate office spaces, Osan (1)", "int_20: Seabees renovate office spaces, Osan (2)", "int_21: A builder marks sheetrock", "int_22: Builders cut sheetrock", "int_23: Piping insulation on ventilation pipes, LIRR Concourse, 2019", "int_24: Seabees small-scale construction project, Jamaica, 2024 (1)", "int_25: Seabees small-scale construction project, Jamaica, 2024 (2)"]
threshold = 0.5 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
lab.compare(photo_a, photo_b, threshold)


In [ ]:
#@title ▶ Step 3b · One site through time { display-mode: "form" }
#@markdown The photos of a series in order, and a chart of each material's share over time.
series = "C" #@param ["C"]
threshold = 0.5 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
lab.series(series, threshold)


> ### 📝 Report question 2
> From Step 3b: which materials rise and which fall over the series, and does that match what a site manager would expect? Give one example where the number changes for a reason that has nothing to do with progress (camera position, sky, an old black-and-white photo...).

## Part 4 · Where does it go wrong?

Three kinds of error to look for: the **words** you use (the model was trained on everyday language, not construction jargon), **weak regions** the model proposes with low confidence, and plain **mistakes** that need a correction. The last step lets you correct the model by drawing a box over what it got wrong.

In [ ]:
#@title ▶ Step 4a · Does the wording matter? { display-mode: "form" }
#@markdown The same material asked for with different words. Alternative wordings are precomputed for int_07, int_13, int_17, int_21; other photos need the live model.
photo = "int_07: House ready for drywall (studs and insulation)" #@param ["int_01: Drywall crane and workers", "int_06: Gypsum board interior wall, Elm Place", "int_07: House ready for drywall (studs and insulation)", "int_09: Drywall penetrations in an industrial building", "int_10: Drywall firestop detail", "int_11: Belchertown: boiler room, 14 Jan 1938", "int_12: Belchertown: plumbing in the basement toilet, 10 Mar 1938", "int_13: Belchertown: tiling in the first-floor toilet, 10 Jun 1938", "int_15: Drywall shaft damage", "int_16: Hanging drywall", "int_17: Carpenters installing sheetrock, LIRR Concourse, 2019", "int_18: Drywall installed in an office hallway, Canaan Valley", "int_19: Seabees renovate office spaces, Osan (1)", "int_20: Seabees renovate office spaces, Osan (2)", "int_21: A builder marks sheetrock", "int_22: Builders cut sheetrock", "int_23: Piping insulation on ventilation pipes, LIRR Concourse, 2019", "int_24: Seabees small-scale construction project, Jamaica, 2024 (1)", "int_25: Seabees small-scale construction project, Jamaica, 2024 (2)"]
material = "studs / framing" #@param ["drywall (plasterboard)", "studs / framing", "insulation", "pipes / ducts", "tiles", "concrete", "ceiling", "floor", "windows / openings", "worker"]
threshold = 0.5 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
lab.phrase_lab(photo, material, threshold)


In [ ]:
#@title ▶ Step 4b · Look at each region and its confidence { display-mode: "form" }
#@markdown Every region the model proposed, numbered, with its confidence. Move the slider to see which ones survive.
photo = "int_07: House ready for drywall (studs and insulation)" #@param ["int_01: Drywall crane and workers", "int_06: Gypsum board interior wall, Elm Place", "int_07: House ready for drywall (studs and insulation)", "int_09: Drywall penetrations in an industrial building", "int_10: Drywall firestop detail", "int_11: Belchertown: boiler room, 14 Jan 1938", "int_12: Belchertown: plumbing in the basement toilet, 10 Mar 1938", "int_13: Belchertown: tiling in the first-floor toilet, 10 Jun 1938", "int_15: Drywall shaft damage", "int_16: Hanging drywall", "int_17: Carpenters installing sheetrock, LIRR Concourse, 2019", "int_18: Drywall installed in an office hallway, Canaan Valley", "int_19: Seabees renovate office spaces, Osan (1)", "int_20: Seabees renovate office spaces, Osan (2)", "int_21: A builder marks sheetrock", "int_22: Builders cut sheetrock", "int_23: Piping insulation on ventilation pipes, LIRR Concourse, 2019", "int_24: Seabees small-scale construction project, Jamaica, 2024 (1)", "int_25: Seabees small-scale construction project, Jamaica, 2024 (2)"]
material = "drywall (plasterboard)" #@param ["drywall (plasterboard)", "studs / framing", "insulation", "pipes / ducts", "tiles", "concrete", "ceiling", "floor", "windows / openings", "worker"]
lab.inspect(photo, material)


In [ ]:
#@title ▶ Step 4c · Correct it with a box { display-mode: "form" }
#@markdown Draw a box over a region that is wrong, click *Submit*: SAM 3 runs again with your box as a *not this* hint. Needs the live model.
photo = "int_07: House ready for drywall (studs and insulation)" #@param ["int_01: Drywall crane and workers", "int_06: Gypsum board interior wall, Elm Place", "int_07: House ready for drywall (studs and insulation)", "int_09: Drywall penetrations in an industrial building", "int_10: Drywall firestop detail", "int_11: Belchertown: boiler room, 14 Jan 1938", "int_12: Belchertown: plumbing in the basement toilet, 10 Mar 1938", "int_13: Belchertown: tiling in the first-floor toilet, 10 Jun 1938", "int_15: Drywall shaft damage", "int_16: Hanging drywall", "int_17: Carpenters installing sheetrock, LIRR Concourse, 2019", "int_18: Drywall installed in an office hallway, Canaan Valley", "int_19: Seabees renovate office spaces, Osan (1)", "int_20: Seabees renovate office spaces, Osan (2)", "int_21: A builder marks sheetrock", "int_22: Builders cut sheetrock", "int_23: Piping insulation on ventilation pipes, LIRR Concourse, 2019", "int_24: Seabees small-scale construction project, Jamaica, 2024 (1)", "int_25: Seabees small-scale construction project, Jamaica, 2024 (2)"]
material = "drywall (plasterboard)" #@param ["drywall (plasterboard)", "studs / framing", "insulation", "pipes / ducts", "tiles", "concrete", "ceiling", "floor", "windows / openings", "worker"]
threshold = 0.5 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
lab.fix(photo, material, threshold)


In [ ]:
#@title ▶ Step 4d · Your own words { display-mode: "form" }
#@markdown Type any phrase: a material, a tool, a machine, a colour. Needs the live model.
photo = "int_19: Seabees renovate office spaces, Osan (1)" #@param ["int_01: Drywall crane and workers", "int_06: Gypsum board interior wall, Elm Place", "int_07: House ready for drywall (studs and insulation)", "int_09: Drywall penetrations in an industrial building", "int_10: Drywall firestop detail", "int_11: Belchertown: boiler room, 14 Jan 1938", "int_12: Belchertown: plumbing in the basement toilet, 10 Mar 1938", "int_13: Belchertown: tiling in the first-floor toilet, 10 Jun 1938", "int_15: Drywall shaft damage", "int_16: Hanging drywall", "int_17: Carpenters installing sheetrock, LIRR Concourse, 2019", "int_18: Drywall installed in an office hallway, Canaan Valley", "int_19: Seabees renovate office spaces, Osan (1)", "int_20: Seabees renovate office spaces, Osan (2)", "int_21: A builder marks sheetrock", "int_22: Builders cut sheetrock", "int_23: Piping insulation on ventilation pipes, LIRR Concourse, 2019", "int_24: Seabees small-scale construction project, Jamaica, 2024 (1)", "int_25: Seabees small-scale construction project, Jamaica, 2024 (2)"]
phrase = "safety helmet" #@param {type:"string"}
threshold = 0.5 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
lab.your_phrase(photo, phrase, threshold)


> ### 📝 Report question 3
> From Step 4a: which wording gave the most sensible mask for the material you chose, and how far apart were the shares? Why would *rebar* and *steel reinforcement bars* give different answers?

> ### 📝 Report question 4
> Describe one mistake you found in Step 4b or 4c (what was included or missed, at which confidence). Did the negative box fix it? What would you tell a colleague who wants to use these percentages in a progress report?

## Part 5 · Quantity take-off on a real foundation plan

These are real drawings (public domain; credits at the bottom). On a drawing the model does not know what a *footing* is: it sees **shapes**. So there are two ways to get quantities out of it: words that describe the shape (*small square*, *circle*) and **boxes** you draw yourself. A box does two things: SAM 3 cuts out the exact outline of what is inside it (its area in pixels), and it can look for **everything else that looks like it** (a count). To turn pixels into square feet you need a **scale**: one box whose real width you know, drawn across a dimensioned bay or along the scale bar.

- **plan_whittier** — Whittier State School hospital (HABS, 1930s): 36 numbered columns on octagonal footings of six sizes. Scale reference: one of the two 12'-0" bays in the middle of the bottom dimension line (12.0 ft). Measure: the footing of column 7, 11, 15 or 19 (detail: 5'-4" across, octagonal, about 23.6 sq ft); the footing of column 6, 10, 14 or 18 (the other interior size); find_all: which columns does it miss?.
- **plan_mill** — Shenandoah-Dives Mill (HAER, CAD drawing): foundation walls, two circular tanks and a graphic scale bar. Scale reference: the graphic scale bar, from 0 to 50 FEET (50.0 ft). Measure: one of the two circular tanks at the bottom left; the hatched circle next to them; try the word 'circle' in Step 5a first.

In [ ]:
#@title ▶ Step 5a · Words on a drawing { display-mode: "form" }
#@markdown Start with *footing*: nothing. Then try *small square* or *circle* and look at what was found: the model sees shapes, not building parts. Count the hits and the misses. Needs the live model.
plan = "plan_whittier" #@param ["plan_whittier", "plan_mill"]
phrase = "footing" #@param {type:"string"}
confidence = 0.4 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.plan_phrase(plan, phrase, confidence)


In [ ]:
#@title ▶ Step 5b · Draw boxes, get areas and counts { display-mode: "form" }
#@markdown Draw the *reference* box first (its width is the scale), then tight boxes labelled *footing*, then *Submit*. With *find_all* ticked, SAM 3 also looks for every element like your first footing box and totals them (blue boxes). Needs the live model.
plan = "plan_whittier" #@param ["plan_whittier", "plan_mill"]
find_all = True #@param {type:"boolean"}
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.takeoff(plan, find_all, confidence)


> ### 📝 Report question 5
> Plan *plan_whittier*: take the scale from a 12'-0" bay on the bottom dimension line. Measure the footing of column 7 (or 11, 15, 19), which the detail gives as 5'-4" across (an octagon of about 23.6 sq ft), and one footing of columns 6-10-14-18. Did *find_all* pick up the octagonal footings? Which columns did it miss and why (look at what is drawn on top of them)?

> ### 📝 Report question 6
> Plan *plan_mill*: use the scale bar (0 to 50 feet) as the reference. Which word finds the two circular tanks in Step 5a? Give their diameter and area in square feet from Step 5b, and explain how you checked the scale (measure the bar twice, or measure a wall whose length you can read).

## Part 6 · Your own photo

In [ ]:
#@title ▶ Your photo, your words { display-mode: "form" }
#@markdown Upload a photo (or open the public link on your phone), type what to find, move the threshold.
#@markdown Test at least 5 photo(s) of your own and take screenshots for your report. Needs the live model.
lab.upload_app()


> ### 📝 Report question 7
> Test 5 photo(s) of your own (walls, floors, a site, a street). For each: the phrase you used, the share measured, and whether the mask is right. What kind of surface or wording failed?

> ### 📝 Report question 8
> Where on a project would a measurement like *share of the photo covered by X* be useful, and where would it mislead? What would you need (camera position, reference lengths, drawings, several photos) to turn it into a real quantity?

## Wrap-up

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
lab.report_summary()


### Photo credits and model
All photos are from Wikimedia Commons under the licence shown with each photo in Step 1a (public domain, CC0, CC BY or CC BY-SA; credits are also in `data/photos/*/credits.json`).

Plan drawings (Part 5), all public domain:
- **plan_whittier**: Whittier State School, Hospital and Receiving Building: foundation plan and details of footings (drawing S1, 1930s). Historic American Buildings Survey (HABS), Library of Congress, copy of original drawing (via Wikimedia Commons). Public domain. https://commons.wikimedia.org/wiki/File:Foundation_Plan_-_Foundation_Plan_and_Details_of_Footing_(drawing_S1)_-_Whittier_State_School,_Hospital_and_Receiving_Building,_11850_East_Whittier_Boulevard,_Whittier,_Los_HABS_CAL,19-WHIT,3-34.tif
- **plan_mill**: Shenandoah-Dives Mill, Silverton, Colorado: foundation plan (HAER CO-91, measured drawing, 1990s). Historic American Engineering Record (HAER), Library of Congress, delineated for HAER (via Wikimedia Commons). Public domain. https://commons.wikimedia.org/wiki/File:Foundation_Plan_-_Shenandoah-Dives_Mill,_135_County_Road_2,_Silverton,_San_Juan_County,_CO_HAER_CO-91_(sheet_7_of_27).png

- Model: SAM 3 by Meta AI (SAM License), loaded from a public mirror of the official checkpoint; a copy of the licence is in `docs/SAM_LICENSE.txt`.
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp4_segmentation`).